In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!unzip -q "archive (2).zip"

In [ ]:
import os

dataset_path = "/content/PlantVillage"

print("Classes Found:\n")

for folder in sorted(os.listdir(dataset_path)):
    print(folder)


Classes Found:

Pepper__bell___Bacterial_spot
Pepper__bell___healthy
Potato___Early_blight
Potato___Late_blight
Potato___healthy
Tomato_Bacterial_spot
Tomato_Early_blight
Tomato_Late_blight
Tomato_Leaf_Mold
Tomato_Septoria_leaf_spot
Tomato_Spider_mites_Two_spotted_spider_mite
Tomato__Target_Spot
Tomato__Tomato_YellowLeaf__Curl_Virus
Tomato__Tomato_mosaic_virus
Tomato_healthy


In [ ]:
import splitfolders

splitfolders.ratio(
    "/content/PlantVillage",
    output="/content/data_split",
    seed=42,
    ratio=(0.8, 0.2)
)

ModuleNotFoundError: No module named 'splitfolders'

In [ ]:
!pip install split-folders


In [ ]:
import splitfolders

splitfolders.ratio(
    "/content/PlantVillage",
    output="/content/data_split",
    seed=42,
    ratio=(0.8, 0.2)
)

Copying files: 20639 files [00:03, 6049.93 files/s]


In [ ]:
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications import EfficientNetB0

from tensorflow.keras.applications.efficientnet import preprocess_input

from tensorflow.keras import layers, models

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

In [ ]:
IMG_SIZE = (224,224)

BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,

    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_gen.flow_from_directory(
    "/content/data_split/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_data = val_gen.flow_from_directory(
    "/content/data_split/val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

Found 16504 images belonging to 15 classes.
Found 4134 images belonging to 15 classes.


In [ ]:
print(train_data.class_indices)

{'Pepper__bell___Bacterial_spot': 0, 'Pepper__bell___healthy': 1, 'Potato___Early_blight': 2, 'Potato___Late_blight': 3, 'Potato___healthy': 4, 'Tomato_Bacterial_spot': 5, 'Tomato_Early_blight': 6, 'Tomato_Late_blight': 7, 'Tomato_Leaf_Mold': 8, 'Tomato_Septoria_leaf_spot': 9, 'Tomato_Spider_mites_Two_spotted_spider_mite': 10, 'Tomato__Target_Spot': 11, 'Tomato__Tomato_YellowLeaf__Curl_Virus': 12, 'Tomato__Tomato_mosaic_virus': 13, 'Tomato_healthy': 14}


In [ ]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        256,
        activation="relu"
    ),

    layers.Dropout(0.4),

    layers.Dense(
        train_data.num_classes,
        activation="softmax"
    )
])

model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │         3,855 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,381,362 (16.71 MB)

 Trainable params: 331,791 (1.27 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_model.h5",
    monitor="val_accuracy",
    save_best_only=True
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[
        early_stop,
        checkpoint
    ]
)

Epoch 1/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 435ms/step - accuracy: 0.6592 - loss: 1.0951

516/516 ━━━━━━━━━━━━━━━━━━━━ 279s 483ms/step - accuracy: 0.7703 - loss: 0.7171 - val_accuracy: 0.8892 - val_loss: 0.3367
Epoch 2/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 397ms/step - accuracy: 0.8767 - loss: 0.3885

516/516 ━━━━━━━━━━━━━━━━━━━━ 212s 411ms/step - accuracy: 0.8855 - loss: 0.3568 - val_accuracy: 0.9156 - val_loss: 0.2530
Epoch 3/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 391ms/step - accuracy: 0.8955 - loss: 0.3139

516/516 ━━━━━━━━━━━━━━━━━━━━ 209s 405ms/step - accuracy: 0.8989 - loss: 0.3013 - val_accuracy: 0.9344 - val_loss: 0.1975
Epoch 4/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 210s 407ms/step - accuracy: 0.9074 - loss: 0.2733 - val_accuracy: 0.9255 - val_loss: 0.2067
Epoch 5/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 212s 412ms/step - accuracy: 0.9175 - loss: 0.2421 - val_accuracy: 0.9332 - val_loss: 0.1943
Epoch 6/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 395ms/step - accuracy: 0.9251 - loss: 0.2220

516/516 ━━━━━━━━━━━━━━━━━━━━ 211s 410ms/step - accuracy: 0.9258 - loss: 0.2187 - val_accuracy: 0.9432 - val_loss: 0.1620
Epoch 7/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 408ms/step - accuracy: 0.9315 - loss: 0.1997

516/516 ━━━━━━━━━━━━━━━━━━━━ 217s 421ms/step - accuracy: 0.9286 - loss: 0.2099 - val_accuracy: 0.9446 - val_loss: 0.1681
Epoch 8/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 217s 420ms/step - accuracy: 0.9294 - loss: 0.2053 - val_accuracy: 0.9388 - val_loss: 0.1719
Epoch 9/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 215s 417ms/step - accuracy: 0.9343 - loss: 0.1891 - val_accuracy: 0.9417 - val_loss: 0.1675
Epoch 10/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 219s 423ms/step - accuracy: 0.9358 - loss: 0.1898 - val_accuracy: 0.9444 - val_loss: 0.1599


In [ ]:
print("Best Validation Accuracy:")
print(max(history.history['val_accuracy']))

Best Validation Accuracy:
0.9446057081222534


In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
fine_history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5,
    callbacks=[
        early_stop,
        checkpoint
    ]
)

Epoch 1/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 281s 483ms/step - accuracy: 0.8436 - loss: 0.4708 - val_accuracy: 0.9190 - val_loss: 0.2316
Epoch 2/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 223s 432ms/step - accuracy: 0.8950 - loss: 0.3081 - val_accuracy: 0.9320 - val_loss: 0.1968
Epoch 3/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 228s 442ms/step - accuracy: 0.9117 - loss: 0.2637 - val_accuracy: 0.9424 - val_loss: 0.1695


In [ ]:
print(max(fine_history.history['val_accuracy']))

0.9424286484718323


In [ ]:
from google.colab import files

files.download("best_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input

model = load_model("best_model.h5")

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving fda63621-b0eb-4938-8ec4-8afffc81ddd6___GH_HL Leaf 268.JPG to fda63621-b0eb-4938-8ec4-8afffc81ddd6___GH_HL Leaf 268.JPG


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input

img_path = list(uploaded.keys())[0]

img = image.load_img(
    img_path,
    target_size=(224,224)
)

img_array = image.img_to_array(img)

img_array = np.expand_dims(
    img_array,
    axis=0
)

img_array = preprocess_input(img_array)

pred = model.predict(img_array)

predicted_class = np.argmax(pred[0])

print("Predicted Class Index:", predicted_class)
print("Confidence:", np.max(pred[0])*100)

1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step
Predicted Class Index: 14
Confidence: 59.274853


In [ ]:
CLASS_NAMES = [
    "Pepper__bell___Bacterial_spot",
    "Pepper__bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite",
    "Tomato__Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato__Tomato_mosaic_virus",
    "Tomato_healthy"
]

print(CLASS_NAMES[predicted_class])

Tomato_healthy


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving eea79799-a869-4a1f-bfa2-5c85a27716ae___RS_Erly.B 8391.JPG to eea79799-a869-4a1f-bfa2-5c85a27716ae___RS_Erly.B 8391.JPG


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input

img_path = list(uploaded.keys())[0]

img = image.load_img(
    img_path,
    target_size=(224,224)
)

img_array = image.img_to_array(img)

img_array = np.expand_dims(
    img_array,
    axis=0
)

img_array = preprocess_input(img_array)

pred = model.predict(img_array)

predicted_class = np.argmax(pred[0])

print("Predicted Class Index:", predicted_class)
print("Confidence:", np.max(pred[0])*100)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Predicted Class Index: 7
Confidence: 84.39472


In [ ]:
CLASS_NAMES = [
    "Pepper__bell___Bacterial_spot",
    "Pepper__bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite",
    "Tomato__Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato__Tomato_mosaic_virus",
    "Tomato_healthy"
]

print(CLASS_NAMES[predicted_class])

Tomato_Late_blight


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving f34d61c3-c2de-4ed5-858b-03c7b21ebcb6___RS_Late.B 5018.JPG to f34d61c3-c2de-4ed5-858b-03c7b21ebcb6___RS_Late.B 5018.JPG


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input

img_path = list(uploaded.keys())[0]

img = image.load_img(
    img_path,
    target_size=(224,224)
)

img_array = image.img_to_array(img)

img_array = np.expand_dims(
    img_array,
    axis=0
)

img_array = preprocess_input(img_array)

pred = model.predict(img_array)

predicted_class = np.argmax(pred[0])

print("Predicted Class Index:", predicted_class)
print("Confidence:", np.max(pred[0])*100)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Predicted Class Index: 10
Confidence: 48.41825


In [ ]:
CLASS_NAMES = [
    "Pepper__bell___Bacterial_spot",
    "Pepper__bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite",
    "Tomato__Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato__Tomato_mosaic_virus",
    "Tomato_healthy"
]

print(CLASS_NAMES[predicted_class])

Tomato_Spider_mites_Two_spotted_spider_mite


In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

val_data.reset()

preds = model.predict(val_data)

y_pred = np.argmax(preds, axis=1)

y_true = val_data.classes

print(
    classification_report(
        y_true,
        y_pred,
        target_names=list(val_data.class_indices.keys())
    )
)

130/130 ━━━━━━━━━━━━━━━━━━━━ 23s 118ms/step
                                             precision    recall  f1-score   support

              Pepper__bell___Bacterial_spot       0.05      0.05      0.05       200
                     Pepper__bell___healthy       0.08      0.10      0.09       296
                      Potato___Early_blight       0.05      0.04      0.05       200
                       Potato___Late_blight       0.07      0.07      0.07       200
                           Potato___healthy       0.03      0.03      0.03        31
                      Tomato_Bacterial_spot       0.10      0.10      0.10       426
                        Tomato_Early_blight       0.00      0.00      0.00       200
                         Tomato_Late_blight       0.12      0.16      0.14       382
                           Tomato_Leaf_Mold       0.06      0.05      0.05       191
                  Tomato_Septoria_leaf_spot       0.09      0.15      0.11       355
Tomato_Spider_mites_

In [ ]:
print(val_data.shuffle)

True


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_data = test_gen.flow_from_directory(
    "/content/data_split/val",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

Found 4134 images belonging to 15 classes.


In [ ]:
preds = model.predict(test_data)

y_pred = np.argmax(preds, axis=1)

y_true = test_data.classes

130/130 ━━━━━━━━━━━━━━━━━━━━ 7s 56ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=list(test_data.class_indices.keys())
    )
)

                                             precision    recall  f1-score   support

              Pepper__bell___Bacterial_spot       0.88      0.86      0.87       200
                     Pepper__bell___healthy       0.76      1.00      0.86       296
                      Potato___Early_blight       1.00      0.83      0.91       200
                       Potato___Late_blight       0.85      0.93      0.89       200
                           Potato___healthy       0.85      0.90      0.88        31
                      Tomato_Bacterial_spot       0.86      0.83      0.84       426
                        Tomato_Early_blight       1.00      0.12      0.21       200
                         Tomato_Late_blight       0.70      0.93      0.80       382
                           Tomato_Leaf_Mold       0.87      0.71      0.78       191
                  Tomato_Septoria_leaf_spot       0.54      0.92      0.68       355
Tomato_Spider_mites_Two_spotted_spider_mite       0.52      0.92

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-60]:
    layer.trainable = False

print("Trainable Layers:")

count = 0

for layer in base_model.layers:
    if layer.trainable:
        count += 1

print(count)

Trainable Layers:
30


In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(
        learning_rate=1e-5
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
fine_history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=8,
    callbacks=[
        early_stop,
        checkpoint
    ]
)

Epoch 1/8
516/516 ━━━━━━━━━━━━━━━━━━━━ 268s 474ms/step - accuracy: 0.7891 - loss: 0.7027 - val_accuracy: 0.8532 - val_loss: 0.4465
Epoch 2/8
516/516 ━━━━━━━━━━━━━━━━━━━━ 219s 424ms/step - accuracy: 0.8502 - loss: 0.4474 - val_accuracy: 0.8858 - val_loss: 0.3395
Epoch 3/8
516/516 ━━━━━━━━━━━━━━━━━━━━ 221s 429ms/step - accuracy: 0.8692 - loss: 0.3891 - val_accuracy: 0.8977 - val_loss: 0.3033


In [ ]:
print("Best Fine Tune Accuracy:")
print(max(fine_history.history["val_accuracy"]))

Best Fine Tune Accuracy:
0.8976777791976929


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("best_model.h5")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_data = test_gen.flow_from_directory(
    "/content/data_split/val",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

Found 4134 images belonging to 15 classes.


In [ ]:
import numpy as np

preds = model.predict(test_data)

y_pred = np.argmax(
    preds,
    axis=1
)

y_true = test_data.classes

130/130 ━━━━━━━━━━━━━━━━━━━━ 24s 119ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=list(
            test_data.class_indices.keys()
        )
    )
)

                                             precision    recall  f1-score   support

              Pepper__bell___Bacterial_spot       0.88      0.86      0.87       200
                     Pepper__bell___healthy       0.76      1.00      0.86       296
                      Potato___Early_blight       1.00      0.83      0.91       200
                       Potato___Late_blight       0.85      0.93      0.89       200
                           Potato___healthy       0.85      0.90      0.88        31
                      Tomato_Bacterial_spot       0.86      0.83      0.84       426
                        Tomato_Early_blight       1.00      0.12      0.21       200
                         Tomato_Late_blight       0.70      0.93      0.80       382
                           Tomato_Leaf_Mold       0.87      0.71      0.78       191
                  Tomato_Septoria_leaf_spot       0.54      0.92      0.68       355
Tomato_Spider_mites_Two_spotted_spider_mite       0.52      0.92

In [ ]:
model.save("plant_model_final.h5")

In [ ]:
from google.colab import files

files.download(
    "plant_model_final.h5"
)

In [ ]:
import random
import os

class_name = random.choice(os.listdir("/content/data_split/val"))

img_name = random.choice(
    os.listdir(f"/content/data_split/val/{class_name}")
)

print("Actual Class:")
print(class_name)

print("Image:")
print(img_name)

Actual Class:
Tomato_Spider_mites_Two_spotted_spider_mite
Image:
29c03d6f-93d8-4dbe-b2f3-456970337a46___Com.G_SpM_FL 1485.JPG


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("best_model.h5")

print(model.input_shape)
print(model.output_shape)

(None, 224, 224, 3)
(None, 15)


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("best_model.h5")

print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │         3,855 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,381,364 (16.71 MB)

 Trainable params: 331,791 (1.27 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 2 (12.00 B)

None


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_data = test_gen.flow_from_directory(
    "/content/data_split/val",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

loss, acc = model.evaluate(test_data)

print("Accuracy:", acc)

Found 4134 images belonging to 15 classes.
130/130 ━━━━━━━━━━━━━━━━━━━━ 23s 103ms/step - accuracy: 0.7591 - loss: 0.8502
Accuracy: 0.7590711116790771


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving best_model.h5 to best_model (1).h5


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("best_model.h5")

loss, acc = model.evaluate(test_data)

print(acc)

130/130 ━━━━━━━━━━━━━━━━━━━━ 26s 114ms/step - accuracy: 0.7591 - loss: 0.8502
0.7590711116790771


In [ ]:
ModelCheckpoint(
    "agrovision_best.keras",
    monitor="val_accuracy",
    save_best_only=True
)

In [ ]:
model = load_model("agrovision_best.keras")
loss, acc = model.evaluate(test_data)

print(acc)

ValueError: File not found: filepath=agrovision_best.keras. Please ensure the file is an accessible `.keras` zip file.

In [ ]:
!find /content -name "*.keras"

In [ ]:
!ls

'archive (2).zip'
'best_model (1).h5'
 best_model.h5
 data_split
'eea79799-a869-4a1f-bfa2-5c85a27716ae___RS_Erly.B 8391.JPG'
'f34d61c3-c2de-4ed5-858b-03c7b21ebcb6___RS_Late.B 5018.JPG'
'fda63621-b0eb-4938-8ec4-8afffc81ddd6___GH_HL Leaf 268.JPG'
 plant_model_final.h5
 plantvillage
 PlantVillage
 sample_data


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("best_model.h5")

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │         3,855 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,381,364 (16.71 MB)

 Trainable params: 331,791 (1.27 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 2 (12.00 B)

In [ ]:
import os

print("best_model.h5")
print(os.path.getsize("best_model.h5") / 1024 / 1024, "MB")

print("\nplant_model_final.h5")
print(os.path.getsize("plant_model_final.h5") / 1024 / 1024, "MB")

best_model.h5
19.704940795898438 MB

plant_model_final.h5
17.169944763183594 MB


In [ ]:
from tensorflow.keras.models import load_model

m1 = load_model("best_model.h5")
m2 = load_model("plant_model_final.h5")

print("Best model params:")
m1.summary()

print("\n\nPlant model params:")
m2.summary()

Best model params:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │         3,855 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,381,364 (16.71 MB)

 Trainable params: 331,791 (1.27 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │         3,855 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,381,364 (16.71 MB)

 Trainable params: 331,791 (1.27 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 2 (12.00 B)

In [ ]:
from tensorflow.keras.models import load_model

m1 = load_model("best_model.h5")
m2 = load_model("plant_model_final.h5")

print("best_model.h5")
print(m1.evaluate(test_data, verbose=0))

print("\nplant_model_final.h5")
print(m2.evaluate(test_data, verbose=0))

best_model.h5
[0.8502246141433716, 0.7590711116790771]

plant_model_final.h5
[0.8502246141433716, 0.7590711116790771]


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("best_model.h5")

In [ ]:
base_model = model.layers[0]

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

print("Fine-tuning ready")

Fine-tuning ready


In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
fine_history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[
        early_stop,
        checkpoint
    ]
)

Epoch 1/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 289s 496ms/step - accuracy: 0.7857 - loss: 0.6708 - val_accuracy: 0.8897 - val_loss: 0.3180
Epoch 2/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 221s 429ms/step - accuracy: 0.8548 - loss: 0.4367 - val_accuracy: 0.9141 - val_loss: 0.2428
Epoch 3/10
516/516 ━━━━━━━━━━━━━━━━━━━━ 222s 429ms/step - accuracy: 0.8759 - loss: 0.3676 - val_accuracy: 0.9286 - val_loss: 0.2037


In [ ]:
print(max(fine_history.history["val_accuracy"]))

0.9286405444145203


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("best_model.h5")

loss, acc = model.evaluate(
    val_data,
    verbose=1
)

print("Accuracy:", acc)

130/130 ━━━━━━━━━━━━━━━━━━━━ 22s 93ms/step - accuracy: 0.7591 - loss: 0.8502
Accuracy: 0.7590711116790771


In [ ]:
from tensorflow.keras.models import load_model

m1 = load_model("best_model.h5")
m2 = load_model("plant_model_final.h5")

print("best_model.h5")
print(m1.evaluate(val_data, verbose=0))

print("\nplant_model_final.h5")
print(m2.evaluate(val_data, verbose=0))

best_model.h5
[0.8502249121665955, 0.7590711116790771]

plant_model_final.h5
[0.8502246737480164, 0.7590711116790771]


In [ ]:
print(max(history.history["val_accuracy"]))

0.9446057081222534


In [ ]:
history.history["val_accuracy"]

[0.8892114162445068,
 0.9155781269073486,
 0.9344460368156433,
 0.9254958629608154,
 0.9332365989685059,
 0.9431543350219727,
 0.9446057081222534,
 0.9388002157211304,
 0.9417029619216919,
 0.9443638324737549]

In [ ]:
from google.colab import files

files.download("best_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>